<a href="https://colab.research.google.com/github/Anushka-1431/PRODIGY_GA_01/blob/main/PRODIGY_GA_01_GPT2_Text_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

In [ ]:
!pip install transformers datasets torch accelerate

In [ ]:
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

from datasets import Dataset

print("Libraries Imported Successfully")

In [ ]:
ai_paragraphs = []

topics = [
    "Artificial Intelligence",
    "Machine Learning",
    "Deep Learning",
    "Neural Networks",
    "Natural Language Processing",
    "Computer Vision",
    "Robotics",
    "Data Science",
    "Generative AI",
    "AI in Healthcare",
    "AI in Education",
    "AI in Finance",
    "AI Ethics",
    "Autonomous Vehicles",
    "Predictive Analytics"
]

for i in range(1000):
    topic = topics[i % len(topics)]

    paragraph = f"""
{topic} is transforming modern industries through intelligent automation and data-driven decision making.
Organizations use {topic.lower()} to improve efficiency, reduce costs, and deliver better customer experiences.
Researchers continue to develop advanced algorithms that increase the accuracy and reliability of AI systems.
The adoption of {topic.lower()} is growing rapidly across healthcare, education, finance, manufacturing, and transportation.
Future innovations in {topic.lower()} are expected to create new opportunities while addressing complex global challenges.
"""
    ai_paragraphs.append(paragraph)

with open("dataset.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(ai_paragraphs))

print("Dataset Created Successfully")
print("Total paragraphs:", len(ai_paragraphs))

In [ ]:
with open("dataset.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("Characters:", len(text))
print(text[:500])

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer Loaded Successfully")

In [ ]:
from transformers import GPT2LMHeadModel

print("GPT2LMHeadModel Imported")

In [ ]:
model = GPT2LMHeadModel.from_pretrained("gpt2")

print("GPT-2 Model Loaded Successfully")

In [ ]:
!ls

In [ ]:
with open("dataset.txt", "r") as f:
    text = f.read()

dataset = Dataset.from_dict({
    "text": [text]
})

print(dataset)

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_dataset = dataset.map(tokenize_function)

print("Tokenization Successful")

In [ ]:
print(tokenized_dataset[0])

In [ ]:
tokenized_dataset = tokenized_dataset.remove_columns(["text"])

tokenized_dataset = tokenized_dataset.map(
    lambda examples: {"labels": examples["input_ids"]},
    batched=True
)

print(tokenized_dataset)

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("Data Collator Created Successfully")

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./gpt2-output",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    save_steps=10,
    save_total_limit=2,
    logging_steps=1,
    report_to="none"
)

print("Training Arguments Created Successfully")

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

print("Trainer Created Successfully")

In [ ]:
trainer.train()

In [ ]:
model.save_pretrained("fine_tuned_gpt2")
tokenizer.save_pretrained("fine_tuned_gpt2")

print("Model Saved Successfully")

In [ ]:
!ls fine_tuned_gpt2

In [ ]:
from transformers import pipeline

# Create text generation pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

# Generate 3 outputs
for i in range(3):
    result = generator(
        "Artificial Intelligence",
        max_new_tokens=50,
        do_sample=True,
        temperature=0.8
    )

    print(f"\n{'='*50}")
    print(f"Generated Output {i+1}")
    print(f"{'='*50}\n")

    print(result[0]["generated_text"])